In [ ]:
#OPTUNA ENGINEERING

import pandas as pd
import numpy as np
import optuna
import json
import warnings
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate

# Keep terminal clean
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def run_ultimate_tuning_lab(stock_num, n_trials=1000):
    try:
        train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
    except FileNotFoundError:
        print(f"[!] Data for Stock {stock_num} not found. Skipping.")
        return None

    X_full = train.drop("target", axis=1)
    y_full = train["target"]
    num_rows = len(X_full)
    apply_log_transform = abs(y_full.skew()) > 1.0

    def objective(trial):
        # ==========================================
        # 🧬 THE GENETIC FEATURE MUTATOR 🧬
        # ==========================================
        X_mutated = X_full.copy()
        base_cols = list(X_mutated.columns)
        
        # 1. Dynamic Interactions (Up to 3 combos)
        for i in range(3):
            if trial.suggest_categorical(f"build_interaction_{i}", [True, False]):
                c1 = trial.suggest_categorical(f"inter_{i}_c1", base_cols)
                c2 = trial.suggest_categorical(f"inter_{i}_c2", base_cols)
                op = trial.suggest_categorical(f"inter_{i}_op", ["add", "sub", "mult", "div"])
                
                col_name = f"{c1}_{op}_{c2}"
                if op == "add": X_mutated[col_name] = X_mutated[c1] + X_mutated[c2]
                elif op == "sub": X_mutated[col_name] = X_mutated[c1] - X_mutated[c2]
                elif op == "mult": X_mutated[col_name] = X_mutated[c1] * X_mutated[c2]
                elif op == "div": X_mutated[col_name] = X_mutated[c1] / (X_mutated[c2] + 1e-9)

        # 2. Non-Linear Expansions (Squares)
        for col in base_cols:
            if trial.suggest_categorical(f"square_{col}", [True, False]):
                X_mutated[f"{col}_squared"] = X_mutated[col] ** 2
                
        # 3. Automated Dropper
        cols_to_drop = []
        for col in X_mutated.columns:
            if trial.suggest_float(f"drop_prob_{col}", 0.0, 1.0) > 0.8:
                cols_to_drop.append(col)
                
        if len(cols_to_drop) < len(X_mutated.columns):
            X_mutated = X_mutated.drop(columns=cols_to_drop)

        # ==========================================
        # 🤖 THE MODEL BUILDER 🤖
        # ==========================================
        scaler_str = trial.suggest_categorical("scaler", ["Standard", "Robust"])
        scaler = StandardScaler() if scaler_str == "Standard" else RobustScaler()
        
        # --- SMALL DATA SAFEGUARD ---
        if num_rows < 500:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso"])
        else:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso", "HistGBM", "XGBoost", "LightGBM"])
        
        if m_type == "Ridge":
            base_model = Ridge(alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True), random_state=42)
        elif m_type == "Lasso":
            base_model = Lasso(alpha=trial.suggest_float("alpha", 1e-4, 1.0, log=True), random_state=42, max_iter=2000)
        elif m_type == "XGBoost":
            base_model = xgb.XGBRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                subsample=trial.suggest_float("subsample", 0.5, 1.0),
                random_state=42, n_jobs=-1,
                tree_method="hist",  # <-- GPU UPGRADE
                device="cuda"        # <-- GPU UPGRADE
            )
        elif m_type == "LightGBM":
            base_model = lgb.LGBMRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                num_leaves=trial.suggest_int("num_leaves", 20, 40),
                random_state=42, n_jobs=-1, verbose=-1,
                device_type="gpu"    # <-- GPU UPGRADE
            )
        else:
            base_model = HistGradientBoostingRegressor(
                max_iter=trial.suggest_int("max_iter", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                random_state=42
            )

        model = TransformedTargetRegressor(regressor=base_model, func=np.log1p, inverse_func=np.expm1) if apply_log_transform else base_model
        pipeline = Pipeline([('scaler', scaler), ('model', model)])

        # ==========================================
        # 📊 TIME-SERIES VALIDATION (No Leakage)
        # ==========================================
        tscv = TimeSeriesSplit(n_splits=5)
        scores = cross_validate(pipeline, X_mutated, y_full, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        return -scores['test_score'].mean()

    # --- REPRODUCIBILITY LOCK ---
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

if __name__ == "__main__":
    print("🚀 INITIATING QUANT LAB: ALL 9 STOCKS (1000 TRIALS) 🚀")
    all_winning_parameters = {}
    
    for stock_num in range(1, 10):
        print(f"\n--- Tuning Stock {stock_num} ---")
        best_params = run_ultimate_tuning_lab(stock_num, n_trials=1000)
        
        if best_params:
            all_winning_parameters[f"stock_{stock_num}"] = best_params
            with open("quant_parameters_final.json", "w") as file:
                json.dump(all_winning_parameters, file, indent=4)
            print(f"[💾] Stock {stock_num} Saved.")

🚀 INITIATING QUANT LAB: ALL 9 STOCKS (200 TRIALS) 🚀

--- Tuning Stock 1 ---


[W 2026-04-01 12:57:41,694] Trial 118 failed with parameters: {'build_interaction_0': True, 'inter_0_c1': 'col_2', 'inter_0_c2': 'col_1', 'inter_0_op': 'div', 'build_interaction_1': False, 'build_interaction_2': True, 'inter_2_c1': 'col_3', 'inter_2_c2': 'col_4', 'inter_2_op': 'sub', 'square_col_0': True, 'square_col_1': False, 'square_col_2': True, 'square_col_3': True, 'square_col_4': True, 'drop_prob_col_0': 0.06693525285775619, 'drop_prob_col_1': 0.0036928082941735897, 'drop_prob_col_2': 0.031284723765826195, 'drop_prob_col_3': 0.938688654478347, 'drop_prob_col_4': 0.6546761734264553, 'drop_prob_col_2_div_col_1': 0.10007168967908331, 'drop_prob_col_3_sub_col_4': 0.5792876894834064, 'drop_prob_col_0_squared': 0.24187486631062835, 'drop_prob_col_2_squared': 0.0924415897293108, 'drop_prob_col_3_squared': 0.2396086767816351, 'drop_prob_col_4_squared': 0.24452078743246927, 'scaler': 'Standard', 'model': 'LightGBM', 'n_estimators': 429, 'learning_rate': 0.0828288308947116, 'max_depth': 5

[💾] Stock 1 Saved.

--- Tuning Stock 2 ---


[W 2026-04-01 12:58:51,472] Trial 14 failed with parameters: {'build_interaction_0': True, 'inter_0_c1': 'col_3', 'inter_0_c2': 'col_10', 'inter_0_op': 'sub', 'build_interaction_1': False, 'build_interaction_2': False, 'square_col_0': True, 'square_col_1': False, 'square_col_2': False, 'square_col_3': True, 'square_col_4': True, 'square_col_5': False, 'square_col_6': True, 'square_col_7': True, 'square_col_8': True, 'square_col_9': False, 'square_col_10': False, 'square_col_11': False, 'square_col_12': False, 'square_col_13': False, 'square_col_14': False, 'drop_prob_col_0': 0.566210571540057, 'drop_prob_col_1': 0.12043853943680342, 'drop_prob_col_2': 0.2873190500666441, 'drop_prob_col_3': 0.6874633014784317, 'drop_prob_col_4': 0.8058871563039826, 'drop_prob_col_5': 0.6495252302661897, 'drop_prob_col_6': 0.5632543552684655, 'drop_prob_col_7': 0.24606828135220843, 'drop_prob_col_8': 0.7029248902678735, 'drop_prob_col_9': 0.9881516957119056, 'drop_prob_col_10': 0.5774099049925319, 'drop_

KeyboardInterrupt: 

In [ ]:
#FINAL SCRIPT

import pandas as pd
import numpy as np
import json
import warnings
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate

warnings.filterwarnings('ignore')

def apply_feature_engineering(df, params):
    """Rebuilds the dataset exactly how Optuna mutated it."""
    X_mutated = df.copy()
    base_cols = list(df.columns)
    
    # 1. Rebuild Interactions
    for i in range(3):
        if params.get(f"build_interaction_{i}", False):
            c1, c2, op = params[f"inter_{i}_c1"], params[f"inter_{i}_c2"], params[f"inter_{i}_op"]
            col_name = f"{c1}_{op}_{c2}"
            if op == "add": X_mutated[col_name] = X_mutated[c1] + X_mutated[c2]
            elif op == "sub": X_mutated[col_name] = X_mutated[c1] - X_mutated[c2]
            elif op == "mult": X_mutated[col_name] = X_mutated[c1] * X_mutated[c2]
            elif op == "div": X_mutated[col_name] = X_mutated[c1] / (X_mutated[c2] + 1e-9)

    # 2. Rebuild Squares
    for col in base_cols:
        if params.get(f"square_{col}", False):
            X_mutated[f"{col}_squared"] = X_mutated[col] ** 2

    # 3. Execute Drops
    cols_to_drop = [c for c in X_mutated.columns if params.get(f"drop_prob_{c}", 0.0) > 0.8]
    if len(cols_to_drop) < len(X_mutated.columns):
        X_mutated = X_mutated.drop(columns=cols_to_drop)
        
    return X_mutated

def build_winning_pipeline(params, apply_log_transform):
    scaler = StandardScaler() if params['scaler'] == "Standard" else RobustScaler()
    m_type = params['model']
    
    if m_type == "Lasso":
        base_model = Lasso(alpha=params['alpha'], random_state=42, max_iter=2000)
    elif m_type == "Ridge":
        base_model = Ridge(alpha=params['alpha'], random_state=42)
    elif m_type == "HistGBM":
        base_model = HistGradientBoostingRegressor(
            max_iter=params['max_iter'], learning_rate=params['learning_rate'], 
            max_depth=params['max_depth'], random_state=42
        )
    elif m_type == "XGBoost":
        base_model = xgb.XGBRegressor(
            n_estimators=params['n_estimators'], learning_rate=params['learning_rate'],
            max_depth=params['max_depth'], subsample=params['subsample'], random_state=42, n_jobs=-1,
            tree_method="hist", device="cuda"  # GPU Enabled
        )
    elif m_type == "LightGBM":
        base_model = lgb.LGBMRegressor(
            n_estimators=params['n_estimators'], learning_rate=params['learning_rate'],
            max_depth=params['max_depth'], num_leaves=params.get('num_leaves', 31), random_state=42, n_jobs=-1, verbose=-1,
            device_type="gpu"  # GPU Enabled
        )
    
    if apply_log_transform:
        return Pipeline([('scaler', scaler), ('model', TransformedTargetRegressor(regressor=base_model, func=np.log1p, inverse_func=np.expm1))])
    return Pipeline([('scaler', scaler), ('model', base_model)])

if __name__ == "__main__":
    try:
        with open("quant_parameters_final.json", "r") as file:
            winning_params = json.load(file)
    except FileNotFoundError:
        print("[!] ERROR: 'quant_parameters_final.json' not found.")
        exit()

    final_output_data = {}
    TARGET_AGGRESSION = 9  

    print("\n" + "="*100)
    print(f"{'STOCK':<8} | {'MODEL':<10} | {'MAE':<7} | {'VOL %':<7} | {'MID':<8} | {'SPREAD':<8} | {'GUARDRAIL':<10}")
    print("-" * 100)

    for stock_num in range(1, 10):
        stock_key = f"stock_{stock_num}"
        if stock_key not in winning_params: continue
            
        params = winning_params[stock_key]
        
        try:
            train = pd.read_csv(f"hackathon_data/stock_{stock_num}_train.csv")
            test = pd.read_csv(f"hackathon_data/stock_{stock_num}_test.csv")
        except FileNotFoundError:
            continue
            
        X_train_raw, y_train_raw = train.drop("target", axis=1), train["target"]
        X_test_raw = test.copy()
        
        # --- THE FLASH CRASH DEFENSE (TARGET CLIPPING) ---
        # Clip the top and bottom 1% of target prices to prevent models from chasing black swan anomalies
        lower_bound = y_train_raw.quantile(0.01)
        upper_bound = y_train_raw.quantile(0.99)
        y_train = y_train_raw.clip(lower=lower_bound, upper=upper_bound)
        
        # 1. Apply Dynamic Feature Engineering
        X_train = apply_feature_engineering(X_train_raw, params)
        X_test = apply_feature_engineering(X_test_raw, params)
        
        # --- THE POISON PILL DEFENSE (DATA SANITIZER) ---
        # Hunt down any Infinity values created by division and turn them into NaNs, 
        # then fill NaNs with 0 so the Scikit-Learn pipeline doesn't crash.
        X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
        X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)
        
        # 2. Build Pipeline
        model_pipeline = build_winning_pipeline(params, abs(y_train.skew()) > 1.0)
        
        # 3. Calculate Risk Diagnostics (Time-Series Split)
        tscv = TimeSeriesSplit(n_splits=5)
        scoring = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}
        cv_results = cross_validate(model_pipeline, X_train, y_train, cv=tscv, scoring=scoring, n_jobs=-1)
        
        mean_mae = (-cv_results['test_mae']).mean()
        mean_rmse = (-cv_results['test_rmse']).mean()
        cv_vol = (-cv_results['test_mae']).std() / mean_mae
        outlier_ratio = mean_rmse / mean_mae
        
        # 4. Strategy Engine & Guardrails
        base_mult = (11 - TARGET_AGGRESSION) / 10.0
        vol_penalty = cv_vol * 5.0
        dynamic_multiplier = min(1.0, max(0.05, base_mult + vol_penalty))
        
        raw_spread_margin = mean_mae * dynamic_multiplier
        
        # HARD GUARDRAILS: Never let the margin drop below £0.50 or exceed £15.00
        spread_margin = np.clip(raw_spread_margin, a_min=0.50, a_max=15.00)
        
        guardrail_status = "ACTIVE" if raw_spread_margin != spread_margin else "OK"
        
        # 5. Final Fit and Predict
        model_pipeline.fit(X_train, y_train)
        mid_preds = model_pipeline.predict(X_test)
        
        # 6. Asymmetric Skew (Momentum Adjustment)
        historical_mean = y_train.tail(50).mean() # Baseline against last 50 known points
        
        predictions_output = []
        for i, pred in enumerate(mid_preds):
            # Shift the bid/ask spread in the direction of the momentum
            momentum_shift = (pred - historical_mean) * 0.10
            skewed_mid = pred + momentum_shift
            
            final_bid = skewed_mid - spread_margin
            final_ask = skewed_mid + spread_margin
            
            predictions_output.append({
                "row": i, 
                "mid": round(float(pred), 4), 
                "bid": round(float(final_bid), 4), 
                "ask": round(float(final_ask), 4)
            })
        
        ex_mid = mid_preds[0]
        print(f"Stock {stock_num:<2} | {params['model']:<10} | {mean_mae:<7.3f} | {cv_vol*100:<6.1f}% | {ex_mid:<8.2f} | {spread_margin*2:<8.3f} | {guardrail_status:<10}")

        final_output_data[stock_key] = {
            "model_metadata": {
                "algorithm": params['model'],
                "diagnostics": {
                    "mae": round(mean_mae, 4),
                    "rmse": round(mean_rmse, 4),
                    "outlier_ratio": round(outlier_ratio, 4),
                    "cv_volatility": round(cv_vol, 4)
                },
                "strategy": {
                    "raw_spread_margin": round(raw_spread_margin, 4),
                    "applied_spread_margin": round(float(spread_margin), 4),
                    "multiplier_used": round(dynamic_multiplier, 4),
                    "features_kept": list(X_train.columns)
                }
            },
            "predictions": predictions_output
        }

    with open("final_market_maker_output.json", "w") as f:
        json.dump(final_output_data, f, indent=4)
        
    print("="*100)
    print(f"\n[✔] Execution Complete. JSON saved to final_market_maker_output.json")